<a href="https://colab.research.google.com/github/dkhan1209/TheAnimalScan/blob/main/TheAnimalScan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
from PIL import Image
import requests
import json
import gradio as gr
import io
import warnings
warnings.filterwarnings('ignore')

print(f'✅ TensorFlow version: {tf.__version__}')
print('✅ Tất cả thư viện đã được import!')

✅ TensorFlow version: 2.20.0
✅ Tất cả thư viện đã được import!


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.model_download("google/efficientnet/tensorFlow1/b4-classification")

print("Path to model files:", path)


100%|██████████| 10.2k/10.2k [00:00<00:00, 4.03MB/s]



100%|██████████| 2.00/2.00 [00:00<00:00, 4.42kB/s]



  0%|          | 0.00/74.3M [00:00<?, ?B/s]



  0%|          | 0.00/3.63M [00:00<?, ?B/s]
  1%|▏         | 1.00M/74.3M [00:01<01:29, 856kB/s]
  3%|▎         | 2.00M/74.3M [00:01<00:45, 1.67MB/s]
  4%|▍         | 3.00M/74.3M [00:01<00:28, 2.64MB/s]

 28%|██▊       | 1.00M/3.63M [00:01<00:03, 908kB/s]
  7%|▋         | 5.00M/74.3M [00:01<00:14, 4.86MB/s]

 55%|█████▌    | 2.00M/3.63M [00:01<00:00, 1.76MB/s]
 11%|█         | 8.00M/74.3M [00:01<00:08, 8.39MB/s]

100%|██████████| 3.63M/3.63M [00:01<00:00, 2.49MB/s]

 13%|█▎        | 10.0M/74.3M [00:01<00:06, 10.3MB/s]
 16%|█▌        | 12.0M/74.3M [00:02<00:05, 12.2MB/s]
 19%|█▉        | 14.0M/74.3M [00:02<00:04, 13.9MB/s]
 22%|██▏       | 16.0M/74.3M [00:02<00:04, 14.9MB/s]
 24%|██▍       | 18.0M/74.3M [00:02<00:03, 16.3MB/s]
 27%|██▋       | 20.0M/74.3M [00:02<00:03, 16.7MB/s]
 31%|███       | 23.0M/74.3M [00:02<00:03, 17.5MB/s]
 35%|███▌      | 26.0M/74.3M [00:02<00:02, 17.2MB/s]
 39%|███▉      | 29.0M/74.3M [00:03<00:02, 19.4MB/s]
 42%|████▏     | 31.0M/74.3M [00:03<00:02, 18.9MB/s

Path to model files: /root/.cache/kagglehub/models/google/efficientnet/tensorFlow1/b4-classification/1


In [ ]:
print('🔄 Đang tải mô hình...')
import requests


model = tf.keras.applications.EfficientNetB0(weights='imagenet')


LABELS_URL = 'https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt'
labels_response = requests.get(LABELS_URL)

imagenet_labels = labels_response.text.strip().split('\n')[1:]

print(f'✅ Đã tải và đồng bộ {len(imagenet_labels)} nhãn với mô hình')
print('✅ Mô hình đã sẵn sàng và chính xác!')

🔄 Đang tải mô hình...
✅ Đã tải và đồng bộ 1000 nhãn với mô hình
✅ Mô hình đã sẵn sàng và chính xác!


In [ ]:

ANIMAL_KEYWORDS = ['dog', 'cat', 'bird', 'fish', 'horse', 'elephant', 'tiger', 'lion', 'bear', 'monkey', 'snake', 'spider', 'butterfly', 'frog', 'turtle', 'shark', 'whale', 'deer', 'rabbit', 'fox', 'wolf', 'owl', 'eagle']

VIET_NAMES = {
    'dog': 'Chó', 'cat': 'Mèo', 'bird': 'Chim', 'golden retriever': 'Chó Golden',
    'tiger': 'Hổ', 'lion': 'Sư tử', 'elephant': 'Voi', 'panda': 'Gấu trúc',
    'hen': 'Gà mái', 'cock': 'Gà trống', 'beagle': 'Chó săn Beagle',
    'rabbit': 'Thỏ', 'goose': 'Ngỗng', 'duck': 'Vịt', 'butterfly': 'Bướm'

}

In [ ]:
def preprocess_image(pil_image):
    """Tiền xử lý ảnh cho mô hình MobileNetV2."""
    img = pil_image.convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

def get_viet_name(english_name):
    """Lấy tên tiếng Việt của động vật."""
    name_lower = english_name.lower()
    for key, viet in VIET_NAMES.items():
        if key in name_lower:
            return viet
    return english_name.replace('_', ' ').title()

def is_animal(label):
    """Kiểm tra nhãn có phải động vật không."""
    label_lower = label.lower()
    return any(kw in label_lower for kw in ANIMAL_KEYWORDS)

def classify_image(pil_image):
    """Nhận diện động vật từ ảnh PIL."""
    if pil_image is None:
        return "⚠️ Vui lòng tải ảnh lên!", None

    try:

        img_tensor = preprocess_image(pil_image)


        predictions = model.predict(img_tensor)
        probs = tf.nn.softmax(predictions[0]).numpy()


        top_indices = np.argsort(probs)[::-1][:10]


        animal_results = []
        all_results = []

        for idx in top_indices:
            if idx < len(imagenet_labels):
                label = imagenet_labels[idx]
                confidence = probs[idx] * 100
                all_results.append((label, confidence))
                if is_animal(label):
                    animal_results.append((label, confidence))


        output_lines = []

        if animal_results:
            top_animal = animal_results[0]
            viet_name = get_viet_name(top_animal[0])
            conf = top_animal[1]


            if conf >= 70:
                confidence_text = f'🟢 Rất chắc chắn ({conf:.1f}%)'
            elif conf >= 40:
                confidence_text = f'🟡 Khá chắc chắn ({conf:.1f}%)'
            else:
                confidence_text = f'🔴 Không chắc chắn ({conf:.1f}%)'

            output_lines.append('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
            output_lines.append(f'🏆 KẾT QUẢ NHẬN DIỆN CHÍNH')
            output_lines.append('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
            output_lines.append(f'   {viet_name}')
            output_lines.append(f'   {confidence_text}')
            output_lines.append(f'   Tên tiếng Anh: {top_animal[0].replace("_", " ").title()}')
            output_lines.append('')

            if len(animal_results) > 1:
                output_lines.append('📋 CÁC KHẢ NĂNG KHÁC:')
                for label, conf in animal_results[1:4]:
                    viet = get_viet_name(label)
                    output_lines.append(f'   • {viet}: {conf:.1f}%')
                output_lines.append('')
        else:
            output_lines.append('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
            output_lines.append('❓ KHÔNG NHẬN DIỆN ĐƯỢC ĐỘNG VẬT')
            output_lines.append('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
            output_lines.append('Ảnh này có thể không phải động vật.')
            output_lines.append('')
            output_lines.append('📋 KẾT QUẢ PHÁT HIỆN ĐƯỢC:')
            for label, conf in all_results[:3]:
                output_lines.append(f'   • {label.replace("_", " ").title()}: {conf:.1f}%')


        chart_data = {}
        results_to_show = animal_results[:5] if animal_results else all_results[:5]
        for label, conf in results_to_show:
            display_name = get_viet_name(label)
            chart_data[display_name] = round(conf, 2)

        return '\n'.join(output_lines), chart_data

    except Exception as e:
        return f'❌ Lỗi: {str(e)}', None

print('✅ Hàm nhận diện đã sẵn sàng!')

✅ Hàm nhận diện đã sẵn sàng!


In [ ]:
# Cập nhật lại hàm tiền xử lý để chuẩn hóa ảnh đúng chuẩn EfficientNet
def preprocess_image_v2(pil_image):
    img = pil_image.convert('RGB')
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32)
    # EfficientNetB0 của tf.keras.applications yêu cầu preprocess này
    img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

# Ghi đè hàm cũ để sử dụng logic mới
def classify_image_fixed(pil_image):
    if pil_image is None: return "⚠️ Vui lòng tải ảnh lên!", None
    try:
        img_tensor = preprocess_image_v2(pil_image)
        predictions = model.predict(img_tensor)
        # EfficientNet đã có softmax ở layer cuối nếu dùng weights='imagenet' chuẩn
        # Nhưng để an toàn ta lấy top k trực tiếp
        top_indices = np.argsort(predictions[0])[::-1][:10]

        animal_results = []
        all_results = []

        for idx in top_indices:
            label = imagenet_labels[idx]
            confidence = predictions[0][idx] * 100
            all_results.append((label, confidence))
            if is_animal(label):
                animal_results.append((label, confidence))

        output_lines = []
        if animal_results:
            top_animal = animal_results[0]
            viet_name = get_viet_name(top_animal[0])
            conf = top_animal[1]
            status = "🟢" if conf > 70 else "🟡" if conf > 40 else "🔴"

            output_lines.extend([
                "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
                "🏆 KẾT QUẢ NHẬN DIỆN CHÍNH",
                "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
                f"   {viet_name}",
                f"   {status} Độ tin cậy: {conf:.1f}%",
                f"   Tên Anh: {top_animal[0].title()}",
                ""
            ])
        else:
            output_lines.append("Không tìm thấy động vật rõ rệt. Kết quả hàng đầu: " + all_results[0][0])

        return "\n".join(output_lines), {}
    except Exception as e:
        return f"❌ Lỗi: {str(e)}", None

# Gán lại hàm cho Gradio sử dụng
classify_image = classify_image_fixed
print('✅ Đã sửa lỗi logic và cập nhật hàm nhận diện!')

✅ Đã sửa lỗi logic và cập nhật hàm nhận diện!


In [ ]:
def predict(image):
    if image is None:
        return '⚠️ Vui lòng tải ảnh lên!'

    try:
        # Chuyển đổi sang PIL Image
        pil_image = Image.fromarray(image.astype('uint8'), 'RGB')

        # Sử dụng hàm classify_image_fixed mà chúng ta đã định nghĩa để xử lý chuẩn
        result_text, _ = classify_image_fixed(pil_image)
        return result_text
    except Exception as e:
        return f'❌ Lỗi xử lý: {str(e)}'

custom_css = """
.gradio-container {
    font-family: 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%) !important;
}
.result-box textarea {
    font-family: 'Courier New', monospace !important;
    font-size: 15px !important;
    background: #0d1117 !important;
    color: #00ff41 !important;
    border: 1px solid #e94560 !important;
}
"""

with gr.Blocks(css=custom_css, title='The Animal Scan') as app:
    gr.HTML("""
    <div style='text-align:center; padding:20px;'>
        <h1 style='color:white;'>🐾 The Animal Scan</h1>
        <p style='color:#ccc;'>Hệ thống nhận diện động vật sử dụng EfficientNetB0</p>
    </div>
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label='📷 Tải ảnh động vật', type='numpy')
            btn = gr.Button('🔍 BẮT ĐẦU NHẬN DIỆN', variant='primary')

        with gr.Column():
            result_text = gr.Textbox(
                label='📊 Kết quả phân tích',
                lines=12,
                elem_classes=['result-box']
            )

    btn.click(fn=predict, inputs=[image_input], outputs=[result_text])

print('🚀 Đang khởi động lại ứng dụng...')
app.launch(share=True, inline=True)

🚀 Đang khởi động lại ứng dụng...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed615904ae5ad05f35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
